# Estruturação confiável de versículos bíblicos

Para elaborar um parser melhor, eu seguiria estas instruções:

- [ ] Defina o modelo de saída antes de extrair: cada registro deve ter livro canônico, capítulo, versículo inicial, versículo final opcional, texto, página do PDF, linha bruta e um indicador de confiança.
- [ ] Extraia preservando contexto de página. Cada fragmento deve carregar o número da página e, idealmente, posição/ordem no PDF, para que qualquer erro seja rastreável até a imagem original.
- [ ] Normalize a fonte com cautela: remova cabeçalho, rodapé e número de página apenas quando coincidirem com padrões confirmados e posição de página; não descarte toda linha numérica genericamente.
- [ ] Reconheça explicitamente os tipos de marcador: capítulo (`nome de livro + capítulo`), versículo simples, intervalo (`2-7`), faixa compacta ou agrupada, títulos, notas e referências cruzadas. Qualquer linha que não se enquadre deve virar texto ou ir para revisão, nunca marcador por suposição.
- [ ] Mantenha intervalos como intervalos. Para `Josué 19:2-7`, prefira guardar um único registro com início 2 e fim 7, ou dividir em `vv. 2–7` somente se o PDF trouxer separação textual confiável. Não invente a distribuição.
- [ ] Valide a sequência em cada capítulo. Registre alertas para repetição, salto, retrocesso, intervalo sobreposto e número improvável; a validação deve apontar `8 → 10 → 10`, não corrigir automaticamente.
- [ ] Tenha regras explícitas para exceções bíblicas: Salmos, livros de um capítulo, títulos de salmos, versículos agrupados, notas e variações de nomenclatura dos livros.
- [ ] Preserve o original e a versão limpa. Nunca substitua o texto bruto: mantenha uma camada de extração, uma de segmentação e outra de normalização, para que uma correção não exija reextrair tudo.
- [ ] Crie uma fila de revisão humana para casos ambíguos. Em vez de decidir silenciosamente, marque registros com baixa confiança, como `2-7` convertido em `27` ou repetições de número.
- [ ] Compare a estrutura final com uma referência canônica de contagem por livro/capítulo. A meta não é substituir o texto da Ave-Maria, mas verificar se as chaves de referência estão completas e coerentes.
- [ ] Faça testes de regressão com páginas problemáticas: `Josué 19`, `2 Reis 3`, páginas com rodapé, quebras por hifenização, início/fim de livro e páginas com títulos. Cada erro corrigido deve virar um caso permanente de teste.
- [ ] Mantenha a base canônica em registros atômicos por versículo, com ID estável (`tradução`, livro, capítulo, versículo), e represente `2-7` como uma referência/ocorrência que aponta para os IDs `2`, `3`, `4`, `5`, `6` e `7`.
- [ ] Se o texto fonte não permitir separar com segurança cada versículo, marque o trecho como `NEEDS_REVIEW`, preservando o intervalo e a fonte bruta, em vez de inventar a divisão.
- [ ] Use uma fonte católica, licenciada e versionada, não apenas "canônica".
- [ ] Valide comparando SQLite e Chroma, garantindo que o índice seja recriado somente a partir da base certificada.

## Critérios de conclusão

- [ ] A extração preserva o original, a versão limpa, a rastreabilidade por página e a fila de revisão humana.
- [ ] A base canônica permanece atômica por versículo, com ocorrências de intervalos mapeadas sem inventar divisão textual.
- [ ] SQLite e Chroma batem exatamente contra a base certificada antes de qualquer publicação ou reindexação.


In [1]:
BIBLIA_INPUT_PDF = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.pdf"
)
BIBLIA_TXT_EXTRACTED = (
    "../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
)
BIBLIA_TEMP = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-temp.txt"
)
BIBLIA_OUTPUT = (
    "../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt"
)

In [2]:
PROCESSAR_PDF = False
PROCESSAR_TXT = True

In [3]:
import pdfplumber


def extrair_texto_pdf(pdf_path, txt_path):
    with pdfplumber.open(pdf_path) as pdf:
        with open(txt_path, "w", encoding="utf-8") as f_out:
            for i, page in enumerate(pdf.pages):
                if i < 2:
                    continue
                elif i == 2904:
                    break
                texto = page.extract_text()
                if texto:
                    f_out.write(texto + "\n")
    print(f"Texto extraído para: {txt_path}")


if PROCESSAR_PDF:
    extrair_texto_pdf(BIBLIA_INPUT_PDF, BIBLIA_TXT_EXTRACTED)

In [4]:
def remover_3_primeiras_linhas(caminho_entrada, caminho_saida):
    with open(caminho_entrada, "r", encoding="utf-8") as f_in:
        linhas = f_in.readlines()
    with open(caminho_saida, "w", encoding="utf-8") as f_out:
        f_out.writelines(linhas[3:])
    print(f"Texto processado para: {caminho_entrada}")


if PROCESSAR_TXT:
    remover_3_primeiras_linhas(BIBLIA_TXT_EXTRACTED, BIBLIA_TEMP)

Texto processado para: ../../data/raw/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt


In [5]:
def remover_trechos_indesejados(caminho):
    trecho_alvo_1 = "Bíblia Ave-Maria"
    trecho_alvo_2 = "© 1959 Editora Ave-Maria."
    remover_linhas = False

    with open(caminho, "r", encoding="utf-8") as f:
        linhas = f.readlines()

    linhas_filtradas = []
    for linha in linhas:
        if trecho_alvo_1 in linha:
            remover_linhas = True
        elif remover_linhas and trecho_alvo_2 in linha:
            continue
        elif remover_linhas and linha.strip().isdigit():
            remover_linhas = False
            continue
        if not remover_linhas:
            linhas_filtradas.append(linha)

    with open(caminho, "w", encoding="utf-8") as f:
        f.writelines(linhas_filtradas)

    print(f"Trechos indesejados removidos de {caminho}")


if PROCESSAR_TXT:
    remover_trechos_indesejados(BIBLIA_TEMP)

Trechos indesejados removidos de ../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-temp.txt


In [6]:
map_proximo_livro = {
    "Livro Inicial": "Gênesis",
    # Antigo Testamento
    "Gênesis": "Êxodo",
    "Êxodo": "Levítico",
    "Levítico": "Números",
    "Números": "Deuteronômio",
    "Deuteronômio": "Josué",
    "Josué": "Juízes",
    "Juízes": "Rute",
    "Rute": "1 Samuel",
    "1 Samuel": "2 Samuel",
    "2 Samuel": "1 Reis",
    "1 Reis": "2 Reis",
    "2 Reis": "1 Crônicas",
    "1 Crônicas": "2 Crônicas",
    "2 Crônicas": "Esdras",
    "Esdras": "Neemias",
    "Neemias": "Tobias",
    "Tobias": "Judite",
    "Judite": "Ester",
    "Ester": "Jó",
    "Jó": "Salmos",
    "Salmos": "1 Macabeus",
    "1 Macabeus": "2 Macabeus",
    "2 Macabeus": "Provérbios",
    "Provérbios": "Eclesiastes",
    "Eclesiastes": "Cântico dos Cânticos",
    "Cântico dos Cânticos": "Sabedoria",
    "Sabedoria": "Eclesiástico",
    "Eclesiástico": "Isaías",
    "Isaías": "Jeremias",
    "Jeremias": "Lamentações de Jeremias",
    "Lamentações de Jeremias": "Baruc",
    "Baruc": "Ezequiel",
    "Ezequiel": "Daniel",
    "Daniel": "Oseias",
    "Oseias": "Joel",
    "Joel": "Amós",
    "Amós": "Abdias",
    "Abdias": "Jonas",
    "Jonas": "Miqueias",
    "Miqueias": "Naum",
    "Naum": "Habacuc",
    "Habacuc": "Sofonias",
    "Sofonias": "Ageu",
    "Ageu": "Zacarias",
    "Zacarias": "Malaquias",
    "Malaquias": "São Mateus",
    # Novo Testamento
    "São Mateus": "São Marcos",
    "São Marcos": "São Lucas",
    "São Lucas": "São João",
    "São João": "Atos",
    "Atos": "Romanos",
    "Romanos": "1 Coríntios",
    "1 Coríntios": "2 Coríntios",
    "2 Coríntios": "Gálatas",
    "Gálatas": "Efésios",
    "Efésios": "Filipenses",
    "Filipenses": "Colossenses",
    "Colossenses": "1 Tessalonicenses",
    "1 Tessalonicenses": "2 Tessalonicenses",
    "2 Tessalonicenses": "1 Timóteo",
    "1 Timóteo": "2 Timóteo",
    "2 Timóteo": "Tito",
    "Tito": "Filemon",
    "Filemon": "Hebreus",
    "Hebreus": "Tiago",
    "Tiago": "1 São Pedro",
    "1 São Pedro": "2 São Pedro",
    "2 São Pedro": "1 São João",
    "1 São João": "2 São João",
    "2 São João": "3 São João",
    "3 São João": "São Judas",
    "São Judas": "Apocalipse",
}

In [ ]:
import re
import os


def transformar_biblia_versiculo_a_versiculo(caminho_entrada, caminho_saida):
    livro_atual = "Livro Inicial"
    capitulo_atual = 0
    versiculo_atual = 0
    buffer_versiculo = []

    # Regex para detectar linhas como "Gênesis 1"
    regex_livro_capitulo = re.compile(r"^(.*)\s+(\d+)$")

    with open(caminho_entrada, "r", encoding="utf-8") as entrada:
        for linha in entrada:
            linha = linha.strip()

            def salvar_versiculo():
                # print(f"Salvando versículo: {livro_atual} {capitulo_atual}:{versiculo_atual} {" ".join(buffer_versiculo).strip()}")
                if versiculo_atual and buffer_versiculo:
                    texto = " ".join(buffer_versiculo).strip()
                    texto = texto.replace("- ", "")  # Remove hífen de divisão de linha
                    saida.write(
                        f"{livro_atual} {capitulo_atual}:{versiculo_atual} {texto}\n"
                    )

            if not linha:
                continue  # pula linhas vazias

            # print(f"Linha atual: {linha}")

            # Verifica se é início de livro (só nome do livro)
            if (
                livro_atual
                != "Apocalipse"  # Apocalipse é o último livro, não há próximo mapeado!
                and linha.startswith(map_proximo_livro[livro_atual])
                and len(linha.split()) <= 3
            ):
                # print(f"Livro encontrado: {linha}")
                salvar_versiculo()
                livro_atual = linha.strip()
                capitulo_atual = 0
                versiculo_atual = 0
                buffer_versiculo.clear()
                continue

            # Verifica se é "Livro Capítulo"
            match = regex_livro_capitulo.match(linha)
            if match:
                # print(f"Capítulo encontrado: {match.group(1)} {match.group(2)}")
                salvar_versiculo()
                capitulo_atual = match.group(2).strip()
                versiculo_atual = 0
                buffer_versiculo.clear()
                continue

            # Verifica se é um número (capítulo ou versículo)
            elif linha.strip().isdigit():
                # print(f"Número encontrado: {linha}")
                salvar_versiculo()
                versiculo_atual = linha.strip()
                buffer_versiculo = []
                continue

            # Se chegou aqui, é texto do versículo
            # print(f"Texto do versículo: {linha}")
            buffer_versiculo.append(linha)

        # Salva o último versículo
        if versiculo_atual and buffer_versiculo:
            salvar_versiculo()

    print(f"Transformação concluída. Versículos salvos em {caminho_saida}")


Transformação concluída. Versículos salvos em ../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible.txt
Arquivo temporário removido: ../../data/processed/biblia/Ave Maria/Portugues-Catolica-AVM-All-Bible-temp.txt


In [ ]:


if PROCESSAR_TXT:
    transformar_biblia_versiculo_a_versiculo(
        BIBLIA_TEMP,
        BIBLIA_OUTPUT,
    )

    if PROCESSAR_TXT and os.path.exists(BIBLIA_TEMP):
        os.remove(BIBLIA_TEMP)
        print(f"Arquivo temporário removido: {BIBLIA_TEMP}")